# Cart-pole control

In the second task you’ll try to craft controllers for the version of the cart-pole problem described by Barto, Sutton, and Anderson in [“Neuronlike Adaptive Elements That Can Solve Difficult Learning Control Problem”.](https://ieeexplore.ieee.org/document/6313077)

<div>
<img src="https://gymnasium.farama.org/_static/videos/mujoco/inverted_pendulum.gif" width="200"/>
</div>



# Gym env
This model has been encoded in the **Inverted Pendulum** environment

Find [here](https://gymnasium.farama.org/environments/mujoco/inverted_pendulum/) a detailed description of the environment
* action space
* observation space
* reward
* model

**The scope of the challenge is to find control laws / policies to set the force acting on the cart. This force must stabilize the pole upward**


In [ ]:
#@title Installing required libraries, setup

%%capture
!pip install gym pyvirtualdisplay
!apt-get install -y xvfb python-opengl ffmpeg
!pip install "gymnasium[mujoco]"
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

# Env and benchmark policies
The following cell defines the environment, `InvertedPendulum-v5` and two baseline policie:
* `zero_policy`: always return 0, that is, no force is applied to the cart
* `random_policy`: a random force is sampled from the admissible intervals of the environment's action space

The effect of these two policies is shown in the following cells via animations of control scenarios.

In [2]:
import gymnasium as gym
import numpy as np
import random
import math
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Create the InvertedPendulum environment (Mujoco must be installed)
env = gym.make("InvertedPendulum-v5", render_mode="rgb_array")

# Random policy - always chose [0]
def zero_policy(obs):
    return [0]

# Random policy
def random_policy(obs):
    return env.action_space.sample()

# Benchmark solutions
The following cells shows animations for the two benchmark policies. The animations halt when the policies are not able to respect the terminal condition (the absolute value of the vertical angle between the pole and the cart is greater than 0.2 radians).  
The cells below run 100 episodes for the `zero_policy` and the `random_policy` policies and retrieve the expected rewards in terms of number of steps before termination. Your solutions should beat both the benchmarks (~24 steps in expectation).

In [3]:
#@title Defining animating function
from gymnasium.wrappers import RecordVideo
import glob
from IPython.display import Video


def animate_policy(policy, env=gym.make("InvertedPendulum-v5", render_mode="rgb_array")):
  recorded = RecordVideo(
      env,                          # this is your AtariPreprocessing+FrameStack env
      video_folder="",
      episode_trigger=lambda ep: True,
      name_prefix="breakout_eval"
  )

  # Run exactly one episode
  obs, info = recorded.reset()
  done = False
  i=0
  obs, _ = env.reset()
  while not done:
      i+=1
      action = policy(obs)
      obs, reward, terminated, truncated, info = recorded.step(action)
      done = terminated or truncated
  recorded.close()

  # Find and embed the just-written MP4
  mp4 = sorted(glob.glob("breakout_eval-episode-*.mp4"))[-1]
  return Video(mp4, embed=True)

In [5]:
animate_policy(random_policy,env=gym.make("InvertedPendulum-v5", render_mode="rgb_array"))

In [6]:
animate_policy(zero_policy,env=gym.make("InvertedPendulum-v5", render_mode="rgb_array"))


In [11]:
def compute_average_reward(policy, env, num_episodes=100):
    total_reward = 0
    for _ in range(num_episodes):
        obs, _ = env.reset()
        done = False
        episode_reward = 0
        while not done:
            action = policy(obs)
            obs, reward, terminated, truncated, info = env.step(action)
            episode_reward += reward
            done = terminated or truncated
        total_reward += episode_reward
    return total_reward / num_episodes

# Create the environment
env = gym.make("InvertedPendulum-v5")

# Compute average reward for zero policy
avg_reward_zero = compute_average_reward(zero_policy, env)
print(f"Average reward for zero policy over 100 episodes: {avg_reward_zero}")

# Compute average reward for random policy
avg_reward_random = compute_average_reward(random_policy, env)
print(f"Average reward for random policy over 100 episodes: {avg_reward_random}")

env.close()

Average reward for zero policy over 100 episodes: 24.65
Average reward for random policy over 100 episodes: 5.43


# Point 1 - Find a reinforcement learning policy

<font color='red'>**Find a reinforcement learning policy stabilising the pole**</font>

For the RL policy you can use one of the following techniques:

- Tabular Q-learning
- Q-learning with function approximation
- DQN

In any case, keep a fixed maximum budget such that the notebook runs in a reasonable amount of time

# Point 2 - Find a policy using LQR/MPC/DP
<font color='red'>**Find a policy belonging to the following classes stabilising the pole**</font>

- LQR with dynamic programming
- MPC

For this task you’ll rely on the linearized equations for this system. Since we are interested in controlling the system close to its equilibrium point (vertical pole), the dynamic equations provided can be linearized around $\theta = 0$.

# Point 3 - Find a policy under uncertainty

For the last task, you will use a different environment, in which a disturbance is added to your policy action:

$a = a + \epsilon$

where $\epsilon \sim \mathcal{N}(\mu = 0, \sigma = 0.25)$

The wrapper below injects an additive disturbance into the action sent to the MuJoCo environment, then clips the result to the original action bounds. This lets us test whether a policy remains stable under actuator noise without changing the policy itself.

<font color='red'>**Either adapt one of the policies you used before or find a new one rejecting the noise.**</font>

In [ ]:
class ActionDisturbanceWrapper(gym.ActionWrapper):
    def __init__(self, env, disturbance_std=0.25, disturbance_bias=0.0, seed=None):
        super().__init__(env)
        self.disturbance_std = disturbance_std
        self.disturbance_bias = disturbance_bias
        self.rng = np.random.default_rng(seed)

    def action(self, action):
        action = np.asarray(action, dtype=np.float32)
        disturbance = self.rng.normal(
            loc=self.disturbance_bias,
            scale=self.disturbance_std,
            size=action.shape,
        )
        disturbed_action = action + disturbance
        return np.clip(disturbed_action, self.action_space.low, self.action_space.high).astype(np.float32)


def make_disturbed_env(render_mode=None, disturbance_std=0.25, disturbance_bias=0.0, seed=None):
    env = gym.make("InvertedPendulum-v5", render_mode=render_mode)
    return ActionDisturbanceWrapper(
        env,
        disturbance_std=disturbance_std,
        disturbance_bias=disturbance_bias,
        seed=seed,
    )